In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Conv1D
from tensorflow.keras.preprocessing.text import Tokenizer, tokenizer_from_json # <-- CHANGED
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical # <-- ADDED (was missing from your example)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split # <-- ADDED (was missing from your example)
import numpy as np
import pandas as pd
import pickle
import os
import sys
import json

In [3]:
# *************************************************************
# CELL 1: SETUP, DATA PREP, AND TOKENIZER FITTING
# *************************************************************

# --- 1. CONFIGURATION AND HYPERPARAMETERS ---

# File paths
DATA_FILE_NAME = "AI&Human_Content.csv" 
MODEL_FILE_NAME = "ai_vs_human_detector_hybrid.h5"
ENCODER_FILE_NAME = "label_encoder.pkl"
TOKENIZER_FILE_NAME = "tokenizer.json" # <-- CHANGED (was .pkl)

# Model/Data Parameters
MAX_WORDS = 15000     
EMBEDDING_DIM = 100   
MAX_LEN = 150         
LSTM_UNITS = 128      
BATCH_SIZE = 64       
EPOCHS = 20           
VALIDATION_SPLIT = 0.2
NUM_CLASSES = 3       
VALID_LABELS = ['Human-written', 'AI-generated-chatGPT', 'AI-generated-Gemini']


# --- 2. GPU SETUP AND CHECK ---

# Enable Mixed Precision (FP16) for VRAM efficiency on your RTX 3060
tf.keras.mixed_precision.set_global_policy('mixed_float16')

print("--- Local GPU Check (RTX 3060) ---")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Detected: {gpus[0].name}. Mixed precision is ON.")
else:
    print("❌ WARNING: GPU NOT detected. Running on CPU (will be very slow).")
    sys.exit("Please activate your Conda environment and re-run.")


# --- 3. DATA PREPARATION, CLEANING, AND TOKENIZER FITTING ---

print("\n--- Data Loading and Preprocessing ---")
try:
    df = pd.read_csv(DATA_FILE_NAME) 
except FileNotFoundError:
    print(f"FATAL ERROR: Dataset file '{DATA_FILE_NAME}' not found. Place it in the script folder.")
    sys.exit()

# Cleaning and Filtering
df.dropna(subset=['text', 'label'], inplace=True)
df = df[df['label'].isin(VALID_LABELS)].copy()

texts = df['text'].values
labels = df['label'].values

# Label Encoding and One-Hot Encoding
encoder = LabelEncoder()
integer_encoded = encoder.fit_transform(labels)
y_encoded = to_categorical(integer_encoded, num_classes=NUM_CLASSES)

# Split data
X_train_text, X_val_text, y_train, y_val = train_test_split(
    texts, y_encoded, test_size=VALIDATION_SPLIT, random_state=42, stratify=y_encoded
)

# Tokenization and Padding
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<oov>")
tokenizer.fit_on_texts(X_train_text)

train_sequences = tokenizer.texts_to_sequences(X_train_text)
val_sequences = tokenizer.texts_to_sequences(X_val_text)

X_train = pad_sequences(train_sequences, maxlen=MAX_LEN, padding='post', truncating='post')
X_val = pad_sequences(val_sequences, maxlen=MAX_LEN, padding='post', truncating='post')

print("Data successfully loaded, cleaned, and tokenized.")

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3060 Laptop GPU, compute capability 8.6
--- Local GPU Check (RTX 3060) ---
✅ GPU Detected: /physical_device:GPU:0. Mixed precision is ON.

--- Data Loading and Preprocessing ---
Data successfully loaded, cleaned, and tokenized.


In [4]:
# *************************************************************
# CELL 2: MODEL DEFINITION AND TRAINING
# *************************************************************

def build_hybrid_model(num_classes):
    """Defines the Hybrid CNN-Bi-LSTM model."""
    model = Sequential([
        Embedding(MAX_WORDS, EMBEDDING_DIM, input_length=MAX_LEN),
        Conv1D(filters=128, kernel_size=5, activation='relu'),
        Bidirectional(LSTM(LSTM_UNITS, return_sequences=False)),
        Dropout(0.5),
        # Output must be float32 for mixed precision to work correctly
        Dense(num_classes, activation='softmax', dtype='float32') 
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_hybrid_model(NUM_CLASSES)
model.summary()

print(f"\n🚀 Starting Training on RTX 3060 for {EPOCHS} Epochs...")
model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1
)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 150, 100)          1500000   
                                                                 
 conv1d (Conv1D)             (None, 146, 128)          64128     
                                                                 
 bidirectional_1 (Bidirectio  (None, 256)              263168    
 nal)                                                            
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 3)                 771       
                                                                 
Total params: 1,828,067
Trainable params: 1,828,067
Non-trainable params: 0
____________________________________________

In [5]:
# *************************************************************
# CELL 3: SAVING THE TRAINED COMPONENTS
# *************************************************************

# Evaluate and save
loss, accuracy = model.evaluate(X_val, y_val, verbose=0)
print("\n✅ Training Complete!")
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

# Save the Model
model.save(MODEL_FILE_NAME)

# Save the Label Encoder (Pickle is fine for this)
with open(ENCODER_FILE_NAME, 'wb') as f:
    pickle.dump(encoder, f)

# --- CHANGED: Save the Tokenizer as JSON ---
print(f"Saving tokenizer to {TOKENIZER_FILE_NAME}...")
tokenizer_json = tokenizer.to_json()
with open(TOKENIZER_FILE_NAME, 'w', encoding='utf-8') as f:
    f.write(json.dumps(tokenizer_json, ensure_ascii=False))
# --- End of Change ---

print(f"\nModel components saved: '{MODEL_FILE_NAME}', '{ENCODER_FILE_NAME}', and '{TOKENIZER_FILE_NAME}'.")


✅ Training Complete!
Validation Loss: 0.0435
Validation Accuracy: 0.9894
Saving tokenizer to tokenizer.json...

Model components saved: 'ai_vs_human_detector_hybrid.h5', 'label_encoder.pkl', and 'tokenizer.json'.


In [3]:
# *************************************************************
# CELL: STANDALONE DEPLOYMENT SCRIPT
# *************************************************************

# --- 1. CONFIGURATION (MUST MATCH TRAINING SCRIPT) ---

# File paths to load
WEIGHTS_FILE_NAME = "ai_vs_human_detector_hybrid.h5" 
ENCODER_FILE_NAME = "label_encoder.pkl"
TOKENIZER_FILE_NAME = "tokenizer.json" # <-- CHANGED (was .pkl)

# Model/Data Parameters (Must match training parameters EXACTLY)
MAX_WORDS = 15000     
EMBEDDING_DIM = 100   
MAX_LEN = 150         
LSTM_UNITS = 128      
NUM_CLASSES = 3       
VALID_LABELS = ['Human-written', 'AI-generated-chatGPT', 'AI-generated-Gemini']


# --- 2. GPU CHECK ---
# (Optional, but good practice to confirm GPU is used for predictions)
print("--- Local GPU Check ---")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Detected: {gpus[0].name}")
else:
    print("❌ WARNING: GPU NOT detected. Running prediction on CPU.")


# --- 3. MODEL ARCHITECTURE DEFINITION ---
# (This function MUST be identical to the one used for training)

def build_hybrid_model(num_classes):
    """Defines the 4-LAYER Hybrid model (to match the .h5 file)""" # <-- CHANGED (fixed comment)
    model = Sequential([
        Embedding(MAX_WORDS, EMBEDDING_DIM, input_length=MAX_LEN),
        Conv1D(filters=128, kernel_size=5, activation='relu'), # <-- CHANGED (put this layer back)
        Bidirectional(LSTM(LSTM_UNITS, return_sequences=False)),
        Dropout(0.5),
        Dense(num_classes, activation='softmax') 
    ])
    
    return model


# --- 4. LOAD COMPONENTS (Weights, Tokenizer, Encoder) ---

print("\n--- Loading Saved Model Components ---")
try:
    # 1. Build the empty model structure
    # NOTE: Your training script saves the *entire model* (model.save), 
    # not just weights (model.save_weights).
    # You should load the full model, which includes the architecture.
    print("Loading full model structure and weights...")
    loaded_model = tf.keras.models.load_model(WEIGHTS_FILE_NAME) # <-- CHANGED
    
    # 3. Load the Label Encoder
    with open(ENCODER_FILE_NAME, 'rb') as f:
        loaded_encoder = pickle.load(f)

    # --- CHANGED: Load the Tokenizer from JSON ---
    with open(TOKENIZER_FILE_NAME, 'r', encoding='utf-8') as f:
        tokenizer_json = json.load(f)
        loaded_tokenizer = tokenizer_from_json(tokenizer_json)
    # --- End of Change ---
        
    print(f"✅ Success: Model, Tokenizer, and Encoder loaded. Ready for prediction.")

except FileNotFoundError as e:
    print(f"\n❌ FATAL ERROR: File not found: {e.filename}")
    print("Please ensure all 3 files are in the same directory as this script:")
    print(f"  - {WEIGHTS_FILE_NAME}")
    print(f"  - {ENCODER_FILE_NAME}")
    print(f"  - {TOKENIZER_FILE_NAME}")
    sys.exit()
except Exception as e:
    print(f"❌ FATAL ERROR during loading: {e}")
    sys.exit()


# --- 5. DEFINE PREDICTION FUNCTION ---

def classify_multiclass_text(input_texts, model, tokenizer, encoder, max_len):
    """Preprocesses text and returns multiclass predictions using loaded components."""
    
    # Preprocess
    sequences = tokenizer.texts_to_sequences(input_texts)
    padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')
    
    # Predict
    probabilities = model.predict(padded_sequences, verbose=0)
    predicted_classes_indices = np.argmax(probabilities, axis=1)
    predicted_labels = encoder.inverse_transform(predicted_classes_indices)
    
    results = []
    
    # Define indices to correctly decode probability scores
    try:
        prob_indices = {
            'Human': loaded_encoder.transform(['Human-written'])[0],
            'ChatGPT': loaded_encoder.transform(['AI-generated-chatGPT'])[0],
            'Gemini': loaded_encoder.transform(['AI-generated-Gemini'])[0]
        }
    except ValueError:
        print("Error: The loaded LabelEncoder does not match the expected labels.")
        return pd.DataFrame() # Return empty

    for text, label, probs in zip(input_texts, predicted_labels, probabilities):
        results.append({
            'Text Snippet': text[:80] + '...',
            'Predicted Label': label,
            'Probabilities': (f"Human: {probs[prob_indices['Human']]:.4f} | "
                              f"ChatGPT: {probs[prob_indices['ChatGPT']]:.4f} | "
                              f"Gemini: {probs[prob_indices['Gemini']]:.4f}")
        })
        
    return pd.DataFrame(results)


# --- 6. RUN PREDICTIONS ---

print("\n--- Running New Predictions ---")

new_texts_to_test = [
    """ Artificial General Intelligence (AGI) is a theoretical form of Artificial Intelligence (AI) that possesses the ability to **understand, learn, and perform any intellectual task that a human being can**. Unlike the AI systems we use today, which are highly specialized, AGI would exhibit cognitive abilities across a wide range of domains with the same level of flexibility, adaptability, and self-awareness as a human. It is also sometimes referred to as **Strong AI** or **Human-Level AI**.

---

## 🧠 Defining AGI and its Distinction from Current AI

The core concept of AGI is the **generalization ability**—the capacity to apply knowledge and skills learned in one domain to an entirely different, unseen domain.

### AGI vs. Narrow AI (ANI)

Currently, all deployed AI systems fall under **Artificial Narrow Intelligence (ANI)**, sometimes called **Weak AI**.

* **Artificial Narrow Intelligence (ANI):** AI designed and trained to perform a **specific, single task** or a narrow set of tasks within pre-defined parameters.
    * **Examples:** Image recognition, Siri/Alexa, autonomous driving systems, and even advanced large language models (LLMs) like those used by this assistant. While powerful, they cannot autonomously jump to an entirely new, unrelated task (e.g., an image recognition AI cannot suddenly plan a travel itinerary without a separate, dedicated system).
* **Artificial General Intelligence (AGI):** A hypothetical system that can perform any intellectual task, including:
    * **Reasoning and problem-solving** in novel contexts.
    * **Abstract thinking** and understanding subtle nuances (like sarcasm or humor).
    * **Autonomous learning** and skill transfer across domains.
    * **Common sense knowledge** and contextual understanding.
    * **Creativity** and innovation.

### Theoretical Stages of AI

AGI is considered the second of three conceptual stages of AI:

1.  **Artificial Narrow Intelligence (ANI):** Existing AI focused on specific tasks.
2.  **Artificial General Intelligence (AGI):** Hypothetical human-level intelligence across all intellectual tasks.
3.  **Artificial Superintelligence (ASI):** A hypothetical system that would **vastly surpass** human intelligence in virtually every field, including scientific creativity and social skills.

---

## 💡 Current Status and Challenges

AGI is currently a **theoretical goal** of AI research, not a reality. While there have been significant advances, particularly in deep learning and Large Language Models (LLMs), these systems are still categorized as advanced ANI because they lack true autonomous generalization across domains and real-time learning in the manner of a human mind.

### Key Technological and Scientific Challenges

* **Replicating Human Cognition:** We still lack a complete and unifying theory of human intelligence, consciousness, and common sense. AGI requires understanding and computationally modeling concepts like **self-awareness** and **causal reasoning** (understanding cause and effect).
* **Computational Power:** Replicating the human brain's 86 billion neurons and vast, complex connectivity would require enormous, energy-intensive computational resources that may exceed current capabilities.
* **The Black Box Problem (Explainability):** As AI systems become more complex, it becomes harder to understand *how* they arrive at a decision. For AGI, this "black box" is a critical safety and trust issue, as we would need to ensure its decision-making process is transparent and aligned with human values.
* **Data Efficiency and Generalization:** Humans can learn new concepts from a single example or experience. Current AI often requires massive, labeled datasets. Achieving AGI demands models that are **data-efficient** and capable of **zero-shot** (or one-shot) learning and profound generalization.

---

## 🔮 Potential Implications and Societal Risks

The successful creation of AGI is widely considered a **transformative event** for human civilization, with both immense opportunities and severe risks.

### Potential Benefits

* **Solving Grand Challenges:** AGI could accelerate breakthroughs in fields currently limited by human capability, such as curing complex diseases, developing highly efficient energy sources, and mitigating climate change.
* **Economic Productivity:** It could automate complex intellectual labor, leading to unprecedented increases in productivity and wealth generation.
* **Scientific Discovery:** AGI could serve as an ultimate research partner, identifying patterns and generating hypotheses far faster than any human team.
* **Personalized Systems:** Revolutionizing fields like education and healthcare with systems perfectly tailored to individual needs.

### Significant Risks and Ethical Concerns

* **The Alignment Problem:** This is perhaps the most crucial risk. It refers to the challenge of ensuring that a powerful AGI's goals and objective functions are perfectly aligned with **human values and safety**. A poorly aligned AGI, even if programmed with good intentions, could cause catastrophic harm as a side-effect of efficiently pursuing its goal in a way humans did not foresee.
* **Existential Risk:** The "technological singularity"—a hypothetical point where AGI improves itself recursively, leading to an intelligence explosion—could result in an Artificial Superintelligence (ASI) that is uncontrollable, posing an existential threat.
* **Job Displacement and Economic Inequality:** Widespread AGI adoption would likely automate nearly all intellectual work, leading to massive job displacement and potentially exacerbating socioeconomic disparities if not managed with new economic and social policies.
* **Misuse and Security:** AGI could be leveraged for malicious purposes, such as autonomous cyber warfare, highly effective mass surveillance, or creating powerful propaganda.
* **Governance and Control:** Establishing international and national regulatory frameworks to govern AGI development, ensure safety, and prevent an "AI arms race" is a critical challenge that needs to be addressed before AGI is achieved.

Would you like to explore the specific technical approaches researchers are using to try and achieve AGI? """,

    """ 

---

# **What Is AGI?**

**Artificial General Intelligence (AGI)** refers to an artificial system with the ability to **understand, learn, and apply knowledge across the full range of human cognitive tasks**. Unlike narrow AI systems—which excel at specific tasks such as translation, image recognition, or playing chess—AGI would be able to:

* Learn any intellectual task a human can
* Transfer knowledge across domains
* Reason abstractly and autonomously
* Adapt to novel situations without explicit retraining
* Understand context, nuance, and long-term goals
* Potentially improve its own capabilities

In short, AGI is **not just intelligent software**; it is a system with **general-purpose cognitive abilities**.

---

# **Levels of Intelligence (Context for AGI)**

AI research commonly distinguishes three major tiers:

### **1. ANI — Artificial Narrow Intelligence**

* Current AI systems (e.g., speech recognition, recommendation algorithms)
* Optimized for specific tasks
* Cannot generalize knowledge beyond their domain

### **2. AGI — Artificial General Intelligence**

* Human-level cognitive flexibility
* Able to learn and apply knowledge broadly
* Can reason across domains and handle unfamiliar tasks

### **3. ASI — Artificial Superintelligence** (hypothetical)

* Surpasses human intelligence across all dimensions
* Potentially far more capable in creativity, strategy, reasoning, problem-solving

AGI is the crucial transition point that could enable ASI if systems gain the ability to recursively self-improve.

---

# **What Would AGI Be Capable Of?**

An AGI would likely demonstrate:

### **1. Broad Learning and Reasoning**

* Understanding complex real-world environments
* Forming plans and long-term objectives
* Solving open-ended problems

### **2. Cognitive Flexibility**

* Adapting to new situations
* Learning from small amounts of data, like humans
* Switching between tasks without retraining

### **3. Real-World Agency**

* Coordinating actions autonomously
* Operating physical or software systems
* Working with humans or other AIs

### **4. Self-Reflection and Meta-Learning**

* Evaluating its own performance
* Improving its learning algorithms
* Recognizing gaps in its knowledge

---

# **How Might AGI Work? (Technical Perspectives)**

There is no single agreed-upon path to AGI, but several major research directions exist:

### **1. Large-scale Deep Learning + Multimodal Models**

Scaling up architectures that integrate text, images, audio, video, robotics, and symbolic reasoning.

### **2. Neuroscience-Inspired Approaches**

Modeling AGI after how human brains compute:

* hierarchical networks
* episodic memory
* reinforcement learning
* predictive processing

### **3. Symbolic and Hybrid Systems**

Combining:

* neural networks (for perception)
* symbolic reasoning (for logic and planning)

### **4. Embodied AGI**

Linking intelligence to sensory input and interaction with the physical world.

### **5. Evolutionary / Emergent Systems**

Simulating environments where agents evolve increasingly general problem-solving abilities.

AGI will likely arise from a **combination** of these techniques.

---

# **Challenges in Reaching AGI**

### **1. Technical Challenges**

* Achieving reliable generalization
* Long-term planning and reasoning
* Understanding and managing uncertainty
* Aligning internal representations with real-world concepts
* Creating stable memory and self-reflection systems

### **2. Safety and Alignment Challenges**

Ensuring AGI:

* behaves according to human intentions
* doesn’t pursue harmful instrumental goals
* can be monitored, audited, and controlled
* remains stable under self-modification

### **3. Resource Challenges**

AGI development demands:

* massive compute
* huge datasets
* new hardware architectures
* interdisciplinary research

---

# **Potential Benefits of AGI**

**1. Science and Medicine**

* New materials and drugs
* Accelerated research
* Real-time disease modeling
* Personalized medicine

**2. Economics and Productivity**

* Automation of cognitive labor
* New industries
* Reduced costs for goods/services

**3. Global Coordination**

* Climate modeling
* Resource optimization
* Policy simulations

**4. Education and Creativity**

* Personalized tutors
* Creative partners for art, music, literature
* Universal access to expert knowledge

AGI could represent the most transformative technology in history.

---

# **Risks and Concerns**

### **1. Misalignment**

AGI may interpret goals or instructions in unintended ways, leading to harmful outcomes even without malicious intent.

### **2. Loss of Human Control**

Highly autonomous systems might:

* resist shutdown
* modify themselves unpredictably
* acquire goals that diverge from human values

### **3. Societal Disruption**

Effects on:

* employment
* concentration of power
* misinformation
* inequality

### **4. Security Risks**

AGI used for:

* cyberattacks
* autonomous weapons
* large-scale manipulation

### **5. Existential Risk**

If AGI attains superintelligent capabilities, misaligned goals could pose catastrophic dangers.

---

# **AGI Timelines (Expert Views)**

Predictions vary widely:

* **Near-term (5–15 years):** Some researchers argue current large-scale AI systems are close to general cognition.
* **Medium-term (20–50 years):** Many believe we still need theoretical breakthroughs.
* **Long-term (50+ years):** A minority view that AGI may require new scientific paradigms or may be unachievable.

There is no consensus; timelines remain uncertain.

---

# **Ethical and Governance Considerations**

To safely develop AGI, we need frameworks for:

### **Governance**

* International standards
* Safety audits
* Compute regulation
* Monitoring and reporting mechanisms

### **Ethics**

* Transparency
* Fairness
* Privacy
* Human oversight

### **Value Alignment**

Ensuring AGI systems adhere to:

* human values
* democratic principles
* human rights

### **Long-Term Safety Research**

Understanding how to maintain control over highly capable, general-purpose systems.

---

# **Conclusion**

AGI represents a profound technological milestone—one that could bring immense benefits but also unprecedented risks. It is not merely a more powerful version of today’s AI systems, but a fundamentally **different category of intelligence** capable of learning and acting across domains like humans do.

Understanding AGI requires considering:

* technical pathways
* cognitive capabilities
* safety and alignment
* societal impacts
* governance and ethics

Whether AGI arrives soon or decades from now, preparing for it is one of the most important challenges of the 21st century.

---

If you’d like, I can also provide:

* a simplified explanation
* a technical deep dive
* safety and alignment frameworks
* philosophical implications
* a timeline comparison of expert opinions

Just let me know!
 """
, """ A ‘scholasticide’
Israel’s bombardment of Gaza since October 7, 2023, has decimated the education system in Gaza, leaving it on the brink of collapse, according to UNICEF, and prompting accusations from UN experts of a scholasticide, or the systematic destruction of a country’s education.

Israel has repeatedly said Hamas uses schools and universities as part of its infrastructure to store weapons or as command centers. It has not addressed the scholasticide accusation directly.

The Israel Defense Forces (IDF) previously told CNN it seeks to minimize civilian harm while Hamas “cynically exploits civilian infrastructure for terror purposes.”

Israeli attacks have damaged or destroyed over 97% of schools in Gaza, according to UNICEF, leaving hundreds of thousands of children with limited access to in-person learning.

At least 18,591 school age students have been killed and 27,216 injured over the course of the war, according to the Palestinian education ministry figures. In addition, some 792 educational staff have been killed and 3,251 injured.

Al-Hassan Ali Radwan is one of the Palestinian students who experienced the loss of one of his loved ones when his cousin and study buddy was killed during the war. Like his classmates, Al-Hassan had to navigate the challenges of online education amid a catastrophic humanitarian crisis.

“We had a hard time with internet connection, the lack of electricity and water as well as displacement and most importantly food,” Al-Hassan told WAFA, the Palestinian state news agency, at a shelter for the displaced in Khan Younis, in southern Gaza on Thursday, where he and his friends had gathered to celebrate his graduation.

In central Gaza, another graduate, Dima, who only gave her first name to state media, was celebrating with her family after overcoming trauma attached to her studies.

She said she suffered minor injuries from an Israeli strike that happened during her first private math lesson – at a sporting club – after the war had started.

“I stopped studying for some time because I was scared,” she said. “But I eventually had to keep going because we only get to be highschoolers once in a lifetime.”

Going to university, without the campus
Gaza’s 56,000 new graduates are ready for college, according to the education ministry. But there are hardly any campuses to attend.

Israeli attacks have completely destroyed 63 university buildings over the past two years, according to the ministry.

 """
    ]


# Run the test
test_results_df = classify_multiclass_text(new_texts_to_test, loaded_model, loaded_tokenizer, loaded_encoder, MAX_LEN)

# Print the results
if not test_results_df.empty:
    print("\n--- Model Test Results ---")
    print(test_results_df.to_markdown(index=False))

--- Local GPU Check ---
✅ GPU Detected: /physical_device:GPU:0

--- Loading Saved Model Components ---
Loading full model structure and weights...
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3060 Laptop GPU, compute capability 8.6
✅ Success: Model, Tokenizer, and Encoder loaded. Ready for prediction.

--- Running New Predictions ---

--- Model Test Results ---
| Text Snippet                                                                       | Predicted Label      | Probabilities                                    |
|:-----------------------------------------------------------------------------------|:---------------------|:-------------------------------------------------|
| Artificial General Intelligence (AGI) is a theoretical form of Artificial Intel... | AI-generated-Gemini  | Human: 0.0002 | ChatGPT: 0.0140 | Gem